# Day 5

## Pytorch Tensor
Pytorch tensor is similar to a numpy array with more capabilities. 
With a tensor, we can move it to a GPU so that we can take advantage of the multiple processors with faster processes. 


In [20]:
import torch

arr = torch.tensor([1, 2, 3])
print(arr)
print(arr.dtype) 
print(arr.shape)
print(arr.device)

arr = arr.to('cuda') # This moves the tensor to GPU
print(arr)
print(arr.device) # this will show it as cuda:0, the 0 here shows that its the first gpu, in case there are many

arr = arr.to('cpu')
print(arr)
print(arr.device)

tensor([1, 2, 3])
torch.int64
torch.Size([3])
cpu
tensor([1, 2, 3], device='cuda:0')
cuda:0
tensor([1, 2, 3])
cpu


### Claude generated prog to test time diff bw CPU and GPU

In [21]:
import torch
import time

size = 1000

# Create two random 1000x1000 matrices on CPU
a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

print(f"Matrices created. Shape: {a_cpu.shape}, dtype: {a_cpu.dtype}, device: {a_cpu.device}")

# --- CPU timing ---
start = time.time()
result_cpu = a_cpu @ b_cpu        # @ is the matmul operator
cpu_time = time.time() - start
print(f"\nCPU time: {cpu_time:.4f} seconds")

# --- GPU timing ---
# Move tensors to GPU
a_gpu = a_cpu.to('cuda')
b_gpu = b_cpu.to('cuda')

print(f"\nMoved to GPU. Device: {a_gpu.device}")

# Warm-up run (first GPU op includes setup time — we throw this away)
_ = a_gpu @ b_gpu
torch.cuda.synchronize()

# Real GPU timing
start = time.time()
result_gpu = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_time = time.time() - start
print(f"GPU time: {gpu_time:.4f} seconds")

# --- The verdict ---
print(f"\n🏁 GPU was {cpu_time / gpu_time:.1f}x faster than CPU")

Matrices created. Shape: torch.Size([1000, 1000]), dtype: torch.float32, device: cpu

CPU time: 0.0073 seconds

Moved to GPU. Device: cuda:0
GPU time: 0.0070 seconds

🏁 GPU was 1.1x faster than CPU


#### Summary of Above Prog

The above program compares the time diff to multiple the matrices bwt CPU and GPU. Firstly the tensor is by default stored in the CPU, only when we manually move it to the GPU using cuda does it move to the GPU. 
We are ignoring the first GPU matmul because the initial setup takes time. factors such as loading cuda libraries, allocating memory may take time. So we ignore the first one and calculate for the second. 
The synchronize part makes python stay in the GPU, usually python will send the data to GPU and move on to next task. Synchronize ensures that it stays there to measure the time.

## Running Distilbert from HF

In [22]:
from transformers import pipeline

pipe = pipeline('sentiment-analysis')

print(pipe('I love this.'))

print(pipe('I am unsure about this. This is pretty bad. But it is good also. But mostly bad.'))


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998741149902344}]
[{'label': 'POSITIVE', 'score': 0.5729963183403015}]


### Above Prog
The above program is from a HF library Transformers. 
The first line is importing the specific pipeline function from transformers package.

Line 2
Here the prog will find default library for sentiment analysis. Thats distilbert. It was made using movie reviews. Movie reviews are usually liike I didnt like or I liked it. They have polarizing rules. So this model can identify if something leans towards positive side or negative. SO it is good for identifying the positive vs negativee.

Line 3
The given line is first tokenized and converted to tokens. The tokens are then sent to distilbert for analysis. It returns the sureity of how certain it is with its prediction. The sentence is pretty basic, so it gives a high certaintity ans.

Line 4
In this line, I have used a lot of unsure wording. I have mixed the sentences with positive/negative and combined them into one line to test the limits of the model. THe model failed here as it could only predict with 57% certainty.

### The whole process one by one. what comes under what and the order

You write:
   pipeline('sentiment-analysis')
        │
        ▼
   transformers library kicks in
        │
        ▼
   It looks up: "What's the default model for sentiment-analysis?"
        │
        ▼
   Answer: distilbert-base-uncased-finetuned-sst-2-english
        │
        ▼
   Goes to huggingface.co and downloads that model
        │
        ▼
   Loads the model into memory
        │
        ▼
   Returns a ready-to-use `pipe` object